In [0]:
%run ../config_setup

# Silver 05 — Maintenance

RDS CDC se `maintenance_record` ke latest states env-configured silver table me SCD2 style maintain karta hai.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, get_json_object, current_timestamp, expr, lit
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()
TARGET_TABLE = f"{SILVER}.maintenance_record"

df = spark.readStream.table(f"{BRONZE}.rds_cdc").filter(col("table") == "maintenance_record").select(
    col("_lsn"), col("_change_type"),
    get_json_object("after", "$.maint_id").alias("maint_id"),
    get_json_object("after", "$.machine_id").alias("machine_id"),
    get_json_object("after", "$.maint_type").alias("maint_type"),
    get_json_object("after", "$.cost").cast("double").alias("cost"),
    get_json_object("after", "$.completed_ts").alias("completed_ts")
)

def upsert_maint(microBatchDF, batchId):
    if microBatchDF.count() == 0: return
    latest = microBatchDF.withColumn("rn", expr("row_number() over (partition by maint_id order by _lsn desc)")).filter("rn = 1").drop("rn")

    if not spark.catalog.tableExists(TARGET_TABLE):
        (latest.filter(col("_change_type") != "delete").drop("_lsn", "_change_type")
               .withColumn("__START_AT", current_timestamp()).withColumn("__END_AT", lit(None).cast("timestamp")).withColumn("__CURRENT", lit(True))
               .write.format("delta").option("overwriteSchema", "true").saveAsTable(TARGET_TABLE))
    else:
        target = DeltaTable.forName(spark, TARGET_TABLE)
        
        deletes = latest.filter(col("_change_type") == "delete")
        target.alias("t").merge(deletes.alias("s"), "t.maint_id = s.maint_id AND t.__CURRENT = true") \
              .whenMatchedUpdate(set={"__END_AT": current_timestamp(), "__CURRENT": lit(False)}).execute()
              
        upserts = latest.filter(col("_change_type") != "delete")
        target.alias("t").merge(upserts.alias("s"), "t.maint_id = s.maint_id AND t.__CURRENT = true") \
              .whenMatchedUpdate(set={"__END_AT": current_timestamp(), "__CURRENT": lit(False)}).execute()
              
        (upserts.drop("_lsn", "_change_type")
                .withColumn("__START_AT", current_timestamp())
                .withColumn("__END_AT", lit(None).cast("timestamp"))
                .withColumn("__CURRENT", lit(True))
                .write.format("delta").mode("append").saveAsTable(TARGET_TABLE))

df.writeStream.foreachBatch(upsert_maint).option("checkpointLocation", f"{CHECKPOINTS}/silver_maint").trigger(availableNow=True).start().awaitTermination()

print("✅ silver_05_maintenance done!")